# Installing Dependencies

# Imports

In [1]:
import sagemaker
import boto3
from sagemaker.sklearn.estimator import SKLearn
import tarfile
import joblib
import json
import io

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


# Configuration

In [ ]:
BUCKET = 'cocktail-ai'
REGION = 'eu-west-1'
DATASET_FILE = 'cocktail_dataset.csv'
ENDPOINT_NAME = 'taste-predictor-dev'
INSTANCE_TYPE = 'ml.m5.large'
SKLEARN_VERSION = '1.2-1'

In [3]:
session = sagemaker.Session()
role = sagemaker.get_execution_role()
print(f"Role: {role}")
print(f"Bucket: {BUCKET}")
print(f"Region: {REGION}")

Role: arn:aws:iam::799518877193:role/cocktail-ai-sagemaker-role
Bucket: cocktail-ai
Region: eu-west-1


In [ ]:
s3 = boto3.client("s3", region_name=REGION)
s3.upload_file(DATASET_FILE, BUCKET, f"data/train/{DATASET_FILE}")
train_uri = f"s3://{BUCKET}/data/train"

# Training Script

Since I am using 9 logistic regression models for fitness evaluation in a Genetic Algorithm Loop, I need to build a custom inference script.

Now the population size is 1000 and it's set to run for 150 generations. I have a total of 9 flavors, so I needed to reduce the net cost, hence I have clubbed all 9 models in a single inference script, instead of 9 seperate endpoints. Essentially a single endpoint will be exposed, which will evaluate the individuals for all the 9 flavors.

One small optimization which I made was in the input for the inference. Since it's nothing but a numpy multiplication, All 1000 individuals are squeezed into a single numpy list, this way per generation we only call the endpoint once, and not a thousaand times. Saving a lot of money! :)

The filename is sagemaker_script.py, which will be our entry point for the estimator.

# Model Training

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

estimator = SKLearn(
    entry_point = "sagemaker_script.py",
    framework_version = SKLEARN_VERSION,
    instance_type = INSTANCE_TYPE,
    role = role,
    output_path = f"s3://{BUCKET}/model-artifacts",
    base_job_name = "cocktail-taste-train",
    sagemaker_session = session,
    enable_sagemaker_metrics = True
)

estimator.fit(inputs={"train": train_uri}, wait=True, logs=True)
print(f"Training job complete")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: cocktail-taste-train-2026-04-12-13-52-21-439


2026-04-12 13:52:22 Starting - Starting the training job...
2026-04-12 13:52:37 Starting - Preparing the instances for training...
2026-04-12 13:52:58 Downloading - Downloading input data...
2026-04-12 13:53:43 Downloading - Downloading the training image......
2026-04-12 13:54:50 Training - Training image download completed. Training in progress.
2026-04-12 13:54:50 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-12 13:54:45,503 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-04-12 13:54:45,507 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-12 13:54:45,510 sa

# Endpoint Deployment

In [12]:
print(estimator.model_data)

s3://cocktail-ai/model-artifacts/cocktail-taste-train-2026-04-12-13-52-21-439/output/model.tar.gz


In [15]:
predictor = estimator.deploy(
    initial_instance_count = 1,
    instance_type = INSTANCE_TYPE,
    endpoint_name = 'taste-predictor-dev',
    serializer = sagemaker.serializers.JSONSerializer(),
    deserializer = sagemaker.deserializers.JSONDeserializer()
)

print(f"Endpoint deployed: taste-predictor-dev")

INFO:sagemaker:Creating model with name: cocktail-taste-train-2026-04-12-13-59-09-121
INFO:sagemaker:Creating endpoint-config with name taste-predictor-dev
INFO:sagemaker:Creating endpoint with name taste-predictor-dev


------!Endpoint deployed: taste-predictor-dev


# Exporting Feature Columns for GA

In [16]:
s3_obj = boto3.client("s3").get_object(Bucket=BUCKET, Key=estimator.model_data.replace(f"s3://{BUCKET}/", ""))
tar_bytes = io.BytesIO(s3_obj["Body"].read())

with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
    joblib_file = tar.extractfile("model.joblib")
    print(joblib_file)
    model = joblib.load(joblib_file)

columns_json = json.dumps(model.feature_columns)
print(f"Feature columns ({len(model.feature_columns)} total) => ", columns_json)

boto3.client("s3").put_object(
    Body=columns_json,
    Bucket=BUCKET,
    Key='ingridients/feature_columns.json',
    ContentType='application/json'
)
print("Successfully uploaded feature columns to: s3")

<ExFileObject name='model.joblib'>
Feature columns (242 total) =>  ["abbott's_bitters_pct", "absinthe_pct", "advocaat_liqueur_pct", "agave_syrup_pct", "aged_jamaican_rum_pct", "aged_rum_(6-10yr)_pct", "agricole_rhum_(unaged)_pct", "almond_milk_liqueur_pct", "amaretto_liqueur_pct", "amaro_(e.g.__nonino)_pct", "amaro_(e.g._averna)_pct", "amaro_(e.g._lucano)_pct", "amaro_(e.g._meletti)_pct", "amaro_(e.g._montenegro)_pct", "amaro_(e.g._ramazzotti)_pct", "amaro_(e.g._santoni)_pct", "ambrato_vermouth_pct", "americano_bianco_pct", "americano_rosso_pct", "amontillado_sherry_pct", "anise_liqueurs_pct", "apple_juice_pct", "apple_schnapps_pct", "applejack_brandy_pct", "apricot_liqueur_pct", "aromatic_bitters_pct", "aromatized_wine_pct", "a\u00f1ejo_tequila_pct", "banana_pct", "banana_liqueur_pct", "basil_leaves_pct", "beer_sparkling_pct", "bianco/blanco_vermouth_pct", "bison_grass_vodka_pct", "bitter_amaro_liqueurs_pct", "bitter_bianco_liqueur_pct", "bitter_fruit_citrus_pct", "bitter_generic_pct"

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


# Deleting the Endpoint

In [17]:
predictor.delete_endpoint()
print(f"Endpoint '{ENDPOINT_NAME}' deleted")

INFO:sagemaker:Deleting endpoint configuration with name: taste-predictor-dev
INFO:sagemaker:Deleting endpoint with name: taste-predictor-dev


Endpoint 'taste-predictor' deleted
